In [1]:
!git clone https://github.com/Break-Through-Tech/Chewy-1A-spot-the-bot-detecting-ai-generated-text.git
%cd Chewy-1A-spot-the-bot-detecting-ai-generated-text/notebooks
!pip -q install datasets huggingface_hub


Cloning into 'Chewy-1A-spot-the-bot-detecting-ai-generated-text'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 160 (delta 74), reused 129 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (160/160), 788.19 KiB | 7.96 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/Chewy-1A-spot-the-bot-detecting-ai-generated-text/notebooks


In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
from spotbot.prepared import load_prepared_splits
from spotbot.evaluation import evaluate_predictions
train_df, val_df, test_df = load_prepared_splits()
print(train_df.shape, val_df.shape, test_df.shape)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Hello-SimpleAI/HC3' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Hello-SimpleAI/HC3' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

HC3.py:   0%|          | 0.00/9.47k [00:00<?, ?B/s]

(65401, 8) (8218, 8) (8151, 8)


In [3]:
import numpy as np, tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

keras.utils.set_random_seed(42)
MAX_LEN = 300
vec = keras.layers.TextVectorization(max_tokens=30000, output_sequence_length=MAX_LEN)
vec.adapt(tf.constant(train_df["text"].astype(str).tolist()))

def encode(df):
    return vec(tf.constant(df["text"].astype(str).tolist())).numpy()

X_train, X_val, X_test = encode(train_df), encode(val_df), encode(test_df)
y_train, y_val, y_test = train_df["label"].values, val_df["label"].values, test_df["label"].values
cw = dict(enumerate(compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)))
print(X_train.shape)


(65401, 300)


In [4]:
def build(dim, pooling, dropout):
    pool = keras.layers.GlobalAveragePooling1D() if pooling == "avg" else keras.layers.GlobalMaxPooling1D()
    m = keras.Sequential([
        keras.layers.Input(shape=(MAX_LEN,)),
        keras.layers.Embedding(30000, dim),
        pool,
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy")
    return m

def run(dim, pooling, dropout):
    keras.utils.set_random_seed(42)
    m = build(dim, pooling, dropout)
    stop = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)
    m.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10,
          batch_size=128, class_weight=cw, callbacks=[stop], verbose=0)
    probs = m.predict(X_val, verbose=0).ravel()
    return m, probs, f1_score(y_val, (probs > 0.5).astype(int), average="macro")

results = {}
for dim in [32, 64, 128]:
    for pooling in ["avg", "max"]:
        for dropout in [0.3, 0.5]:
            m, probs, f1 = run(dim, pooling, dropout)
            results[(dim, pooling, dropout)] = (f1, m, probs)
            print(dim, pooling, dropout, round(f1, 4))


32 avg 0.3 0.9782
32 avg 0.5 0.9778
32 max 0.3 0.9672
32 max 0.5 0.967
64 avg 0.3 0.976
64 avg 0.5 0.9783
64 max 0.3 0.9733
64 max 0.5 0.9739
128 avg 0.3 0.9771
128 avg 0.5 0.9771
128 max 0.3 0.9762
128 max 0.5 0.9771


In [5]:
best_cfg = max(results, key=lambda k: results[k][0])
f1_best, model, val_probs = results[best_cfg]
print("best:", best_cfg, round(f1_best, 4))

thresholds = np.arange(0.3, 0.71, 0.05)
best_t = max(thresholds, key=lambda t: f1_score(y_val, (val_probs > t).astype(int), average="macro"))
print("best threshold:", round(best_t, 2))

val_pred = (val_probs > best_t).astype(int)
scores = evaluate_predictions(val_df["label"], val_pred, val_df["source"])
print(scores["macro_f1"], scores["accuracy"])
print(scores["confusion_matrix"])
print(scores["per_domain"])


best: (64, 'avg', 0.5) 0.9783
best threshold: 0.5
0.9782571722899736 0.9810172791433439
[[5495   92]
 [  64 2567]]
{'finance': {'macro_f1': 0.970232601395862, 'accuracy': 0.9704142011834319, 'per_class':        precision    recall        f1  support
class                                        
human   0.976684  0.959288  0.967908      393
ai      0.965142  0.980088  0.972558      452, 'confusion_matrix': array([[377,  16],
       [  9, 443]])}, 'medicine': {'macro_f1': 0.9920307340614681, 'accuracy': 0.9920318725099602, 'per_class':        precision  recall        f1  support
class                                      
human   1.000000   0.984  0.991935      125
ai      0.984375   1.000  0.992126      126, 'confusion_matrix': array([[123,   2],
       [  0, 126]])}, 'open_qa': {'macro_f1': 0.8933311974900158, 'accuracy': 0.9129511677282378, 'per_class':        precision    recall        f1  support
class                                        
human   0.760000  0.957983  0.847584     

In [6]:
test_probs = model.predict(X_test, verbose=0).ravel()
test_pred = (test_probs > best_t).astype(int)
test_scores = evaluate_predictions(test_df["label"], test_pred, test_df["source"])
print(test_scores["macro_f1"], test_scores["accuracy"])
display(test_scores["per_class"])
print(test_scores["confusion_matrix"])
print(test_scores["per_domain"])


0.9790791133154189 0.9817200343516133


,precision,recall,f1,support
class,,,,
human,0.988750,0.984285,0.986512,5536
ai,0.967045,0.976291,0.971646,2615


[[5449   87]
 [  62 2553]]
{'finance': {'macro_f1': 0.9749863295903964, 'accuracy': 0.9750889679715302, 'per_class':        precision    recall        f1  support
class                                        
human   0.972152  0.974619  0.973384      394
ai      0.977679  0.975501  0.976589      449, 'confusion_matrix': array([[384,  10],
       [ 11, 438]])}, 'medicine': {'macro_f1': 0.9883424956698547, 'accuracy': 0.9883720930232558, 'per_class':        precision    recall        f1  support
class                                        
human   1.000000  0.975806  0.987755      124
ai      0.978102  1.000000  0.988930      134, 'confusion_matrix': array([[121,   3],
       [  0, 134]])}, 'open_qa': {'macro_f1': 0.9326072738647027, 'accuracy': 0.9472573839662447, 'per_class':        precision    recall        f1  support
class                                        
human   0.850746  0.957983  0.901186      119
ai      0.985294  0.943662  0.964029      355, 'confusion_matrix': array([